# Problem 1: LSTM/GRU for Sequence Classification

## Speech Emotion Recognition using Recurrent Neural Networks (PyTorch)

---

## Problem Statement

**Goal**: Classify emotional content in speech by treating audio features as a **time series** and using recurrent networks to capture temporal dependencies.

### Why This Architecture?
- Speech is inherently **sequential** - emotions unfold over time
- LSTMs/GRUs can capture **long-term dependencies** (e.g., rising anger throughout an utterance)
- Handles **variable-length** audio clips naturally

### Input/Output Specification
- **Input**: MFCC sequence of shape `(batch_size, time_steps, n_mfcc)` where `n_mfcc=39` (13 MFCCs + 13 deltas + 13 delta-deltas)
- **Output**: Softmax probabilities over 8 emotion classes

### Architecture Overview
```
Input: MFCC Sequence (T, 39)
    |
    v
Bidirectional LSTM (128 units) -- captures past AND future context
    |
    v
Bidirectional LSTM (64 units)
    |
    v
Attention Layer (optional) -- focus on emotionally salient frames
    |
    v
Dense (64, ReLU) + Dropout
    |
    v
Dense (8, Softmax) --> Emotion Probabilities
```

## 1. Setup and Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import os
import glob
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
Device: cpu


## 2. Data Loading and Parsing

### RAVDESS Filename Structure
Format: `Modality-VocalChannel-Emotion-Intensity-Statement-Repetition-Actor.wav`

- **Emotion codes**: 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised

In [3]:
DATA_DIR = Path("data")

EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

def parse_ravdess_filename(filepath):
    """Extract metadata from RAVDESS filename."""
    name = Path(filepath).stem
    parts = name.split("-")
    if len(parts) != 7:
        return None
    
    actor_id = int(parts[6])
    return {
        "file_path": filepath,
        "emotion_code": parts[2],
        "emotion_label": EMOTION_MAP.get(parts[2]),
        "actor_id": parts[6],
        "gender": "male" if actor_id % 2 == 1 else "female",
    }

# Load all wav files
wav_paths = sorted(glob.glob(str(DATA_DIR / "Actor_*" / "*.wav")))
print(f"Found {len(wav_paths)} .wav files")

# Build dataframe
rows = [parse_ravdess_filename(p) for p in wav_paths]
rows = [r for r in rows if r is not None]
df = pd.DataFrame(rows)

print(f"Dataset shape: {df.shape}")
print(f"\nEmotion distribution:")
print(df['emotion_label'].value_counts())

Found 1440 .wav files
Dataset shape: (1440, 5)

Emotion distribution:
emotion_label
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96
Name: count, dtype: int64


## 3. Feature Extraction

### MFCC Features for LSTM

For LSTM input, we need a **sequence** of feature vectors:
- Extract MFCCs at each time frame
- Include delta and delta-delta coefficients for dynamics
- Pad/truncate sequences to fixed length for batching

**Feature Glossary**:
- **MFCC (0-12)**: Voice timbre/quality fingerprint
- **Delta MFCC**: How MFCCs change over time (speech dynamics)
- **Delta-Delta MFCC**: Acceleration of MFCC changes

In [4]:
# Hyperparameters for feature extraction
SAMPLE_RATE = 22050
N_MFCC = 13
MAX_LEN = 200  # Max time steps (frames)
HOP_LENGTH = 512

def extract_mfcc_sequence(file_path, sr=SAMPLE_RATE, n_mfcc=N_MFCC, max_len=MAX_LEN):
    """
    Extract MFCC sequence with deltas for LSTM input.
    
    Returns: (time_steps, features) array where features = n_mfcc * 3
    """
    try:
        y, _ = librosa.load(file_path, sr=sr)
        
        # Extract MFCCs
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=HOP_LENGTH)
        
        # Compute deltas
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        
        # Stack features: (n_mfcc*3, time_steps)
        features = np.vstack([mfcc, delta, delta2])
        
        # Transpose to (time_steps, features)
        features = features.T
        
        # Pad or truncate to max_len
        if features.shape[0] < max_len:
            pad_width = max_len - features.shape[0]
            features = np.pad(features, ((0, pad_width), (0, 0)), mode='constant')
        else:
            features = features[:max_len]
        
        return features
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

print(f"Feature shape per sample: ({MAX_LEN}, {N_MFCC * 3})")
print(f"Total features per frame: {N_MFCC * 3} (MFCC + Delta + Delta-Delta)")

Feature shape per sample: (200, 39)
Total features per frame: 39 (MFCC + Delta + Delta-Delta)


In [5]:
# Extract features for all files
print("Extracting MFCC sequences...")

X = []
y = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    features = extract_mfcc_sequence(row['file_path'])
    if features is not None:
        X.append(features)
        y.append(row['emotion_label'])

X = np.array(X)
y = np.array(y)

print(f"\nFeatures shape: {X.shape}")
print(f"Labels shape: {y.shape}")

Extracting MFCC sequences...


100%|██████████| 1440/1440 [00:34<00:00, 41.23it/s]


Features shape: (1440, 200, 39)
Labels shape: (1440,)


## 4. Data Preprocessing

In [6]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

n_classes = len(label_encoder.classes_)
print(f"Classes: {label_encoder.classes_}")
print(f"Number of classes: {n_classes}")

# Split data (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)

print(f"\nTrain: {X_train.shape[0]} samples")
print(f"Val: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
Number of classes: 8

Train: 979 samples
Val: 173 samples
Test: 288 samples


In [7]:
# Normalize features (fit on train only)
n_train, n_timesteps, n_features = X_train.shape

scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, n_features)
scaler.fit(X_train_flat)

X_train_norm = scaler.transform(X_train_flat).reshape(n_train, n_timesteps, n_features)
X_val_norm = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)
X_test_norm = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

print(f"Normalized train mean: {X_train_norm.mean():.4f}")
print(f"Normalized train std: {X_train_norm.std():.4f}")

Normalized train mean: 0.0000
Normalized train std: 1.0000


## 5. PyTorch Dataset and DataLoader

In [8]:
class EmotionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = EmotionDataset(X_train_norm, y_train)
val_dataset = EmotionDataset(X_val_norm, y_val)
test_dataset = EmotionDataset(X_test_norm, y_test)

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 31
Val batches: 6
Test batches: 9


## 6. Model Architecture

### Bidirectional LSTM with Attention

In [9]:
class Attention(nn.Module):
    """Simple attention mechanism to weight time steps."""
    
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)
    
    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden_size)
        attn_weights = torch.softmax(self.attention(lstm_output), dim=1)
        # Weighted sum
        context = torch.sum(attn_weights * lstm_output, dim=1)
        return context, attn_weights


class LSTMEmotionClassifier(nn.Module):
    """
    Bidirectional LSTM model for emotion classification.
    """
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3, use_attention=True):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.use_attention = use_attention
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention layer
        if use_attention:
            self.attention = Attention(hidden_size * 2)  # *2 for bidirectional
        
        # Classification head
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(64, 32)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(32, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        # lstm_out: (batch, seq_len, hidden_size * 2)
        
        if self.use_attention:
            context, _ = self.attention(lstm_out)
        else:
            # Use last hidden state
            context = lstm_out[:, -1, :]
        
        # Classification
        x = F.relu(self.fc1(context))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        
        return x


# Create model
INPUT_SIZE = N_MFCC * 3  # 39 features
HIDDEN_SIZE = 128
NUM_LAYERS = 2

model = LSTMEmotionClassifier(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=n_classes,
    dropout=0.3,
    use_attention=True
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

LSTMEmotionClassifier(
  (lstm): LSTM(39, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): Attention(
    (attention): Linear(in_features=256, out_features=1, bias=True)
  )
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (dropout1): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=32, out_features=8, bias=True)
)

Total parameters: 587,369


## 7. Training

In [10]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()
    
    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
    
    return total_loss / len(loader), correct / total

In [11]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

NUM_EPOCHS = 100
PATIENCE = 15
best_val_loss = float('inf')
patience_counter = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_lstm_model.pt')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(torch.load('best_lstm_model.pt'))
print("\nTraining complete. Best model loaded.")

Starting training...
Epoch 5/100 - Train Loss: 1.7877, Train Acc: 0.2748 - Val Loss: 1.6708, Val Acc: 0.2890


KeyboardInterrupt: 

## 8. Training Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('lstm_training_curves.png', dpi=150)
plt.show()

## 9. Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Get predictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - LSTM Emotion Classifier')
plt.tight_layout()
plt.savefig('lstm_confusion_matrix.png', dpi=150)
plt.show()

## 10. GRU Variant (Comparison)

In [ ]:
class GRUEmotionClassifier(nn.Module):
    """Bidirectional GRU model for comparison."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        gru_out, _ = self.gru(x)
        # Use last hidden state
        x = F.relu(self.fc1(gru_out[:, -1, :]))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Uncomment to train GRU model
# gru_model = GRUEmotionClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, n_classes).to(device)
# print(gru_model)

## 11. Key Takeaways

### Strengths of LSTM/GRU for SER
- Captures **temporal evolution** of emotions naturally
- **Bidirectional** processing leverages full context
- **Attention** can highlight key emotional moments

### Limitations
- Sequential processing = slower training than CNNs
- May struggle with very long sequences
- Sensitive to hyperparameters (layers, units, dropout)

### Next Steps
- Try CNN-LSTM hybrid (extract local features first)
- Experiment with different attention mechanisms
- Use pre-trained audio embeddings (OpenL3, VGGish)